# 3 · Labeling (CUSUM event sampling + triple barrier)

Entries on M15. Event candidates picked via CUSUM filter on log returns, threshold scaled by M15 ATR%
(vol-adaptive, no cross-TF merge). Triple barrier labels each event: upper/lower = entry ± k×ATR,
vertical = N bars ahead.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd

from config.settings import DATA_PROCESSED, DATA_FEATURES, DATA_LABELS

SYMBOL = "GBPUSD"
TF = "M15"

raw = pd.read_parquet(DATA_PROCESSED / f"{SYMBOL}_{TF}.parquet")
feat = pd.read_parquet(DATA_FEATURES / f"{SYMBOL}_{TF}_features.parquet")

df = raw.merge(feat[["datetime", "atr", "atr_pct"]], on="datetime", how="inner").reset_index(drop=True)
print(f"{TF}: {len(df)} rows  ({df['datetime'].iloc[0]} .. {df['datetime'].iloc[-1]})")
df.tail()

M15: 356499 rows  (2012-01-11 01:30:00+00:00 .. 2026-07-10 00:00:00+00:00)


,datetime,open,high,low,close,volume,spread,split,atr,atr_pct
356494,2026-07-09 23:00:00+00:00,1.34118,1.34125,1.34116,1.34123,164.70,0.0,test,0.000570,0.000425
356495,2026-07-09 23:15:00+00:00,1.34121,1.34121,1.34106,1.34108,112.95,0.0,test,0.000542,0.000404
356496,2026-07-09 23:30:00+00:00,1.34108,1.34127,1.34106,1.34126,70.65,0.0,test,0.000518,0.000386
356497,2026-07-09 23:45:00+00:00,1.34126,1.34132,1.34112,1.34132,214.65,0.0,test,0.000495,0.000369
356498,2026-07-10 00:00:00+00:00,1.34131,1.34133,1.34087,1.34089,1609.21,0.0,test,0.000493,0.000367


## CUSUM filter

Symmetric CUSUM on log returns of `close`. Threshold `h_t = k * atr_pct_t` — adaptive to local volatility.
Causal: `atr_pct` at bar `t` only uses bars `<= t`, and the running sums only look backward, so no lookahead.

In [2]:
def cusum_filter(returns: pd.Series, threshold: pd.Series) -> pd.DatetimeIndex:
    """Symmetric CUSUM event sampler (Lopez de Prado). threshold indexed same as returns."""
    s_pos, s_neg = 0.0, 0.0
    events = []
    idx = returns.index
    for i in range(1, len(idx)):
        t = idx[i]
        r = returns.iat[i]
        h = threshold.iat[i]
        if np.isnan(r) or np.isnan(h):
            continue
        s_pos = max(0.0, s_pos + r)
        s_neg = min(0.0, s_neg + r)
        if s_pos > h:
            s_pos = 0.0
            events.append(t)
        elif s_neg < -h:
            s_neg = 0.0
            events.append(t)
    return pd.DatetimeIndex(events)


K_CUSUM = 1.0  # threshold multiplier on atr_pct

log_ret = np.log(df["close"]).diff()
threshold = K_CUSUM * df["atr_pct"]

log_ret.index = df["datetime"]
threshold.index = df["datetime"]

event_times = cusum_filter(log_ret, threshold)
print(f"events: {len(event_times)}  ({len(event_times) / len(df):.2%} of bars)")
event_times[:10]

events: 88646  (24.87% of bars)


DatetimeIndex(['2012-01-11 06:00:00+00:00', '2012-01-11 08:00:00+00:00',
               '2012-01-11 08:45:00+00:00', '2012-01-11 09:30:00+00:00',
               '2012-01-11 10:15:00+00:00', '2012-01-11 11:15:00+00:00',
               '2012-01-11 12:00:00+00:00', '2012-01-11 12:15:00+00:00',
               '2012-01-11 13:15:00+00:00', '2012-01-11 14:30:00+00:00'],
              dtype='datetime64[ms, UTC]', freq=None)

## Triple barrier labeling

For each event: entry = `close` at event bar. Upper/lower barriers = entry ± `k_barrier * atr` (atr at event bar).
Vertical barrier = `max_hold` bars ahead. Scan forward `high`/`low` — whichever barrier touched first wins;
label 1 (upper), -1 (lower), 0 (vertical/timeout, i.e. no barrier hit in time).

In [3]:
K_BARRIER = 2.0   # upper/lower barrier width, multiples of ATR
MAX_HOLD = 16      # vertical barrier, bars ahead (16 * 15m = 4h)


def triple_barrier(df: pd.DataFrame, event_times: pd.DatetimeIndex, k: float, max_hold: int) -> pd.DataFrame:
    idx_of = {t: i for i, t in enumerate(df["datetime"])}
    close = df["close"].values
    high = df["high"].values
    low = df["low"].values
    atr = df["atr"].values
    dt = df["datetime"].values
    n = len(df)

    rows = []
    for t in event_times:
        i = idx_of.get(t)
        if i is None or np.isnan(atr[i]):
            continue

        entry_price = close[i]
        upper = entry_price + k * atr[i]
        lower = entry_price - k * atr[i]
        end = min(i + max_hold, n - 1)

        label = 0
        exit_i = end
        exit_reason = "vertical"
        for j in range(i + 1, end + 1):
            if high[j] >= upper:
                label, exit_i, exit_reason = 1, j, "upper"
                break
            if low[j] <= lower:
                label, exit_i, exit_reason = -1, j, "lower"
                break

        rows.append({
            "datetime": t,
            "entry_price": entry_price,
            "upper_barrier": upper,
            "lower_barrier": lower,
            "exit_datetime": dt[exit_i],
            "exit_price": close[exit_i],
            "bars_held": exit_i - i,
            "label": label,
            "exit_reason": exit_reason,
        })

    return pd.DataFrame(rows)


labels = triple_barrier(df, event_times, K_BARRIER, MAX_HOLD)
print(labels["label"].value_counts())
labels.tail()

label
 1    34054
-1    33599
 0    20993
Name: count, dtype: int64


,datetime,entry_price,upper_barrier,lower_barrier,exit_datetime,exit_price,bars_held,label,exit_reason
88641,2026-07-09 17:15:00+00:00,1.34143,1.342937,1.339923,2026-07-09 21:00:00,1.33955,15,-1,lower
88642,2026-07-09 19:15:00+00:00,1.34102,1.342258,1.339782,2026-07-09 21:00:00,1.33955,7,-1,lower
88643,2026-07-09 21:00:00+00:00,1.33955,1.340684,1.338416,2026-07-09 21:45:00,1.34013,3,1,upper
88644,2026-07-09 21:30:00+00:00,1.34042,1.341736,1.339104,2026-07-10 00:00:00,1.34089,10,0,vertical
88645,2026-07-09 22:00:00+00:00,1.34091,1.342345,1.339475,2026-07-10 00:00:00,1.34089,8,0,vertical


## Save

In [4]:
path = DATA_LABELS / f"{SYMBOL}_{TF}_labels.parquet"
labels.to_parquet(path, index=False)
print(f"[SAVE] {path.relative_to(DATA_LABELS.parent.parent)}: {len(labels)} rows, {labels.shape[1]} cols")

[SAVE] data/labels/GBPUSD_M15_labels.parquet: 88646 rows, 9 cols
